# Day 1 · 02 · Lakehouse, Delta Lake & Trusted Gold Layer

![Medallion to Power BI](../../assets/images/medallion_to_powerbi.png)

The team now understands the SQL Warehouse, the tools and the source data
grain. The next problem is architectural: where should governed reporting
logic live? This notebook explains the Lakehouse and Delta Lake foundations,
then builds the first Kimball-style Gold layer that Power BI can trust.

## Business Scenario

RetailHub still has three problems: different reports calculate revenue
differently, report performance changes depending on the visual and filter,
and nobody can point to one approved source of truth. The answer is to build
a governed Gold layer in Databricks and make Power BI a thin reporting layer.

## Learning Objectives

By the end of this notebook you can:

- explain Lakehouse DW foundations and why Delta Lake matters for BI,
- inspect Delta metadata with `DESCRIBE DETAIL` and `DESCRIBE HISTORY`,
- describe Bronze, Silver and Gold responsibilities,
- explain where Medallion fits next to Kimball, Inmon, Data Vault and Data Mesh,
- build a first Kimball-style Gold layer,
- explain why Gold is a BI contract, not only a table,
- check Gold tables with reconciliation and quality gates before reporting.

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this notebook.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before the demo starts.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream setup or demo notebook has not created the required tables.

In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: data/generate_training_dataset.ipynb")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"data/generate_training_dataset.ipynb\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## 9. Lakehouse DW Overview: why this is not a classic warehouse

The Lakehouse gives BI teams warehouse-like reliability without forcing all
data into a proprietary warehouse engine. The key difference is that governed
SQL, BI, streaming and ML workloads can use the same Delta tables through
Unity Catalog, while compute is separated by workload.

| Area | Classic Data Warehouse | Databricks Lakehouse |
|---|---|---|
| Storage format | proprietary | open Delta/Parquet with transaction log |
| Compute | often tightly coupled to storage | SQL Warehouse, jobs and notebooks scale independently |
| Data types | mostly structured | structured, semi-structured and unstructured |
| Reliability | strong transactions inside the DW | Delta ACID transactions on object storage |
| Governance | warehouse permissions | Unity Catalog across tables, views, volumes and lineage |
| BI contract | dimensional marts | Gold layer tables/views/aggregates |

For Power BI developers the important point is simple: Gold should feel like a
trusted warehouse mart, but it is built on open Delta tables in the Lakehouse.

## 10. Delta Lake essentials for analysts

Every Silver and Gold table used in this course is a Delta table. Analysts do
not need to administer Delta internals, but they should understand the features
that make a Gold layer reliable enough for reporting.

| Delta capability | Why it matters for BI |
|---|---|
| ACID transactions | reports do not read half-written refreshes |
| schema enforcement | bad writes fail instead of corrupting reports |
| schema evolution | new columns can be added intentionally and audited |
| transaction log | every table change has history and version metadata |
| time travel | a KPI can be compared to a previous table version |
| `MERGE INTO` | incremental updates and CDC-style changes are atomic |
| `DESCRIBE DETAIL` | file count, size, format and location are inspectable |
| `DESCRIBE HISTORY` | refresh operations can be audited |
| `OPTIMIZE` / clustering | physical layout can improve repeated BI queries |
| `VACUUM` | storage cleanup affects time-travel retention |

In [ ]:
for obj in [f"{SILVER}.order_lines", f"{GOLD}.fact_sales"]:
    print("Delta detail:", obj)
    detail = spark.sql(f"DESCRIBE DETAIL {obj}")
    visible_cols = [c for c in ["format", "numFiles", "sizeInBytes", "location"] if c in detail.columns]
    detail.select(*visible_cols).show(truncate=False)

print("Recent history for Gold fact table")
history = spark.sql(f"DESCRIBE HISTORY {GOLD}.fact_sales LIMIT 10")
visible_history_cols = [c for c in ["version", "timestamp", "operation", "operationMetrics"] if c in history.columns]
history.select(*visible_history_cols).show(truncate=False)

### Safe time-travel orientation

Time travel is useful for auditing and troubleshooting: "what did this table
look like before the last refresh?" It is not a backup strategy. If old files
are removed by `VACUUM`, older versions may no longer be queryable.

In [ ]:
history_versions = [
    r["version"]
    for r in spark.sql(f"DESCRIBE HISTORY {GOLD}.fact_sales")
             .select("version")
             .orderBy("version")
             .limit(1)
             .collect()
]

if history_versions:
    oldest_available_version = history_versions[0]
    print("Oldest visible Delta version:", oldest_available_version)
    spark.sql(f"""
    SELECT COUNT(*) AS rows_at_version
    FROM {GOLD}.fact_sales VERSION AS OF {oldest_available_version}
    """).show()
else:
    print("No Delta history version was returned for this table.")

### Delta contract demo: schema evolution and constraints

Delta is not only a storage format. It is also a contract surface for BI. This
small disposable table shows three concepts from the data engineering course in
an analyst-friendly way:

| Concept | What to notice | BI relevance |
|---|---|---|
| schema evolution | a new column is added intentionally | semantic models should react to planned schema changes, not surprises |
| table constraint | an invalid row is rejected | bad facts do not silently enter dashboards |
| operation history | table changes are auditable | refresh issues can be explained with versions and operations |

The table is created from a small sample and dropped at the end of the cell. It
is safe to run during a live demo.


In [ ]:
DEMO_CONTRACT_TABLE = f"{GOLD}.delta_contract_demo"

try:
    spark.sql(f"DROP TABLE IF EXISTS {DEMO_CONTRACT_TABLE}")

    spark.sql(f"""
    CREATE TABLE {DEMO_CONTRACT_TABLE}
    USING DELTA
    AS
    SELECT
      line_id,
      order_id,
      product_id,
      order_date,
      quantity,
      line_revenue
    FROM {GOLD}.fact_sales
    WHERE quantity > 0 AND line_revenue IS NOT NULL
    LIMIT 1000
    """)

    spark.sql(f"""
    ALTER TABLE {DEMO_CONTRACT_TABLE}
    ADD CONSTRAINT positive_quantity CHECK (quantity > 0)
    """)

    spark.sql(f"ALTER TABLE {DEMO_CONTRACT_TABLE} ADD COLUMNS (contract_note STRING)")
    spark.sql(f"UPDATE {DEMO_CONTRACT_TABLE} SET contract_note = 'validated demo sample'")

    print("Delta detail for disposable contract table")
    detail = spark.sql(f"DESCRIBE DETAIL {DEMO_CONTRACT_TABLE}")
    detail.select("format", "numFiles", "sizeInBytes").show(truncate=False)

    print("Trying to insert a row that violates the positive_quantity constraint")
    try:
        spark.sql(f"""
        INSERT INTO {DEMO_CONTRACT_TABLE}
        SELECT
          line_id,
          order_id,
          product_id,
          order_date,
          CAST(-1 AS INT) AS quantity,
          line_revenue,
          'invalid demo row' AS contract_note
        FROM {DEMO_CONTRACT_TABLE}
        LIMIT 1
        """)
        print("[WARN] Invalid row was inserted. Check runtime support for Delta constraints.")
    except Exception as constraint_error:
        print("[OK] Delta constraint rejected the invalid write.")
        print(type(constraint_error).__name__, str(constraint_error)[:240])

    print("Recent history for disposable contract table")
    history = spark.sql(f"DESCRIBE HISTORY {DEMO_CONTRACT_TABLE} LIMIT 10")
    visible_history_cols = [c for c in ["version", "timestamp", "operation", "operationParameters"] if c in history.columns]
    history.select(*visible_history_cols).show(truncate=False)

except Exception as e:
    print("[INFO] Delta contract demo could not be executed on this runtime/warehouse.")
    print("Keep the markdown as concept orientation and continue with the Gold modeling cells.")
    print(type(e).__name__, str(e)[:300])
finally:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {DEMO_CONTRACT_TABLE}")
        print("[OK] Disposable Delta contract table cleaned up.")
    except Exception as cleanup_error:
        print("[INFO] Cleanup skipped or failed.", type(cleanup_error).__name__, str(cleanup_error)[:200])


### Delta operations to recognize, not overuse in BI notebooks

| Operation | Where it belongs | Trainer guidance |
|---|---|---|
| `MERGE INTO` | Silver/Gold incremental pipelines | explain as the atomic upsert pattern |
| `OPTIMIZE` | scheduled maintenance or predictive optimization | do not run blindly in every demo notebook |
| `ZORDER BY` | older/manual layout tuning | useful conceptually; Liquid Clustering is preferred for many new tables |
| Liquid Clustering | new large tables with evolving filters | mention as modern layout strategy |
| `VACUUM` | lifecycle cleanup | never use aggressive retention casually; it affects time travel |
| constraints | data-quality enforcement | useful when rules are stable and table-level |
| Predictive Optimization | UC managed table maintenance | helps automate compaction and cleanup decisions |

For this analyst course, `DESCRIBE DETAIL`, `DESCRIBE HISTORY`, query profile
and Gold table grain are the Delta topics participants should be comfortable
using live.

## 11. Medallion Architecture

Bronze, Silver and Gold are not just quality labels. They are responsibilities.

| Layer | Main responsibility | Typical consumer | Should contain |
|---|---|---|---|
| Bronze | raw landing and traceability | data engineering | raw files/tables, ingestion metadata |
| Silver | cleaned and conformed entities | analytics engineering | reusable business entities |
| Gold | curated analytics contracts | BI, dashboards, Genie | stable facts, dimensions, KPI aggregates |

	Gold should not be a dumping ground for every possible column. It should be a
	clear contract for a defined analytical use case.
	

## 12. Where Medallion fits among architecture patterns

![Architecture patterns to Power BI Gold](../../assets/images/architecture_patterns_to_powerbi.png)

Power BI developers often hear multiple architecture terms in the same
conversation: Medallion, Kimball, Inmon, Data Vault and Data Mesh. They are not
all competing answers to the same question.

| Pattern | Main idea | What a Power BI developer should remember |
|---|---|---|
| Medallion | Bronze/Silver/Gold quality and consumption layers in the lakehouse | practical Databricks layering; Power BI usually consumes Gold |
| Kimball | dimensional model with facts and dimensions | best default mental model for BI reporting and slicers |
| Inmon | enterprise data warehouse with an integrated, normalized core | useful enterprise modelling idea; often upstream from BI marts |
| Data Vault | historized integration model: hubs, links, satellites, business vault | strong for audit/history and many sources; Power BI should not usually query Raw Vault directly |
| Data Mesh | operating model: domain ownership and data products | helps define who owns Gold objects, contracts and quality |

Key point: **Medallion is a lakehouse layering pattern. Kimball, Inmon and Data
Vault are modelling approaches. Data Mesh is an operating model.** In this
training, Gold is where these ideas become a usable BI contract.

### Decision guide for this training

| Situation | Good default |
|---|---|
| Building a Power BI report source | Gold Kimball-style fact/dim or aggregate |
| Integrating many operational systems with full audit history | Data Vault upstream, then publish BI-friendly Gold |
| Creating a central enterprise model | Inmon-style integrated core can inform Silver/Gold design |
| Assigning ownership and quality responsibility | Data Mesh data product thinking |
| Teaching Databricks Lakehouse flow | Medallion from Bronze to Silver to Gold |

For this course we keep the hands-on path simple: Medallion explains the layers,
Kimball shapes the BI-facing Gold model, and Data Mesh gives us ownership and
contract language.

## 13. Gold as a BI contract

![Gold business value](../../assets/images/gold_business_value.png)

A BI contract answers practical questions:

- What is the grain?
- Which filters define each KPI?
- Which dimensions are supported?
- Which rows are excluded?
- Who owns the definition?
- What is the refresh expectation?
- Which object should Power BI query?

If this is not written in Databricks, the logic usually leaks into many Power
BI reports and becomes impossible to govern.

## 14. Kimball-style Gold model

![Kimball Gold model](../../assets/images/kimball_gold_model.png)

We use a simple star-schema pattern:

- dimensions describe filtering and grouping,
- facts store measurable events,
- aggregates support fast dashboard pages,
- views can support parameterized or incremental access patterns.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_date
COMMENT 'Date dimension for RetailHub BI reports.'
AS
SELECT DISTINCT
  order_date AS date_key,
  year(order_date) AS year,
  quarter(order_date) AS quarter,
  month(order_date) AS month,
  date_format(order_date, 'yyyy-MM') AS year_month,
  dayofweek(order_date) AS day_of_week
FROM {SILVER}.sales_orders
WHERE order_date IS NOT NULL
""")

spark.sql(f"SELECT * FROM {GOLD}.dim_date ORDER BY date_key LIMIT 10").show()

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_product
COMMENT 'Product dimension for BI slicers and category analysis.'
AS
SELECT
  product_id,
  product_name,
  category,
  subcategory,
  unit_cost
FROM {SILVER}.products
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_customer
COMMENT 'Customer dimension for BI slicers and regional analysis.'
AS
SELECT
  customer_id,
  customer_name,
  segment,
  region AS customer_region
FROM {SILVER}.customers
""")

print("[OK] Dimensions refreshed")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales
COMMENT 'Gold sales fact at order-line grain. One row per sales order line.'
AS
SELECT
  l.line_id,
  l.order_id,
  o.customer_id,
  l.product_id,
  o.order_date,
  o.status,
  o.channel,
  l.quantity,
  l.unit_price,
  p.unit_cost,
  CASE WHEN o.status = 'Completed' THEN true ELSE false END AS is_completed,
  l.quantity * l.unit_price AS line_revenue,
  l.quantity * (l.unit_price - p.unit_cost) AS line_margin
FROM {SILVER}.order_lines l
JOIN {SILVER}.sales_orders o ON l.order_id = o.order_id
LEFT JOIN {SILVER}.products p ON l.product_id = p.product_id
""")

spark.sql(f"SELECT * FROM {GOLD}.fact_sales LIMIT 10").show()

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard
COMMENT 'BI-ready detail table joining sales fact with conformed dimensions.'
AS
SELECT
  f.line_id,
  f.order_id,
  f.order_date,
  d.year,
  d.quarter,
  d.month,
  d.year_month,
  f.status,
  f.channel,
  f.is_completed,
  f.quantity,
  f.unit_price,
  f.unit_cost,
  f.line_revenue,
  f.line_margin,
  c.customer_id,
  c.customer_name,
  c.segment,
  c.customer_region,
  p.product_id,
  p.product_name,
  p.category,
  p.subcategory
FROM {GOLD}.fact_sales f
LEFT JOIN {GOLD}.dim_date d ON f.order_date = d.date_key
LEFT JOIN {GOLD}.dim_customer c ON f.customer_id = c.customer_id
LEFT JOIN {GOLD}.dim_product p ON f.product_id = p.product_id
""")

spark.sql(f"SELECT COUNT(*) AS rows FROM {GOLD}.fact_sales_dashboard").show()

## 15. Reconciliation and quality gates

![Gold quality gate](../../assets/images/gold_quality_gate.png)

Gold tables should be checked before they become report sources. The checks
below are intentionally simple and readable for analysts.

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer,
  SUM(CASE WHEN unit_price IS NULL THEN 1 ELSE 0 END) AS missing_price
FROM {GOLD}.fact_sales_dashboard
""").show()

In [ ]:
spark.sql(f"""
WITH fact_total AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM {GOLD}.fact_sales
),
dashboard_total AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard
)
SELECT
  f.revenue AS fact_revenue,
  d.revenue AS dashboard_revenue,
  ROUND(f.revenue - d.revenue, 2) AS diff
FROM fact_total f CROSS JOIN dashboard_total d
""").show()

### Business impact of bad data escaping to Gold

Data quality is not an academic concern. The numbers below show what a KPI dashboard would report if bad data slipped through vs. what a governed Gold layer reports after applying quality rules.

This is the business case for the Gold contract: a 0.3–0.5% revenue discrepancy at €50M annual sales is €150K–€250K — enough to invalidate a bonus calculation or a region performance review.

In [ ]:
spark.sql(f"""
WITH silver_raw AS (
  SELECT ROUND(SUM(line_revenue), 2) AS revenue
  FROM {SILVER}.order_lines
  WHERE status = 'Completed'
),
silver_dirty_issues AS (
  SELECT
    ROUND(SUM(CASE WHEN order_date > current_date() THEN line_revenue ELSE 0 END), 2) AS future_dated,
    ROUND(SUM(CASE WHEN unit_price IS NULL THEN 0 ELSE 0 END), 2) AS missing_price_impact,
    COUNT(CASE WHEN order_date > current_date() THEN 1 END) AS future_dated_rows,
    COUNT(CASE WHEN unit_price IS NULL THEN 1 END) AS missing_price_rows
  FROM {SILVER}.order_lines
  WHERE status = 'Completed'
),
gold_clean AS (
  SELECT ROUND(SUM(line_revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard
  WHERE is_completed
)
SELECT
  s.revenue AS silver_completed_revenue,
  g.revenue AS gold_completed_revenue,
  ROUND(s.revenue - g.revenue, 2) AS overcount,
  ROUND(100.0 * (s.revenue - g.revenue) / NULLIF(s.revenue, 0), 4) AS overcount_pct,
  i.future_dated_rows,
  i.future_dated AS future_dated_revenue
FROM silver_raw s CROSS JOIN gold_clean g CROSS JOIN silver_dirty_issues i
""").show(truncate=False)

print("""
Discussion:
- Silver 'Completed' revenue includes future-dated orders (seeded quality issue).
- Gold quality filter excludes them: the difference is the business error that would appear in a dashboard
  if analysts connected Power BI directly to Silver instead of to Gold.
- At scale this gap compounds: promotions, bonuses, and supply chain decisions are made on KPI numbers.
  A 0.1% overcount in €50M revenue = €50K that never happened.
""")

In [ ]:
quality = spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS missing_order_date,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product_id,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer_id
FROM {GOLD}.fact_sales_dashboard
""").first()

assert quality["rows"] > 0, "Gold dashboard table is empty"
assert quality["rows"] == quality["distinct_lines"], "Gold dashboard grain is not unique at line_id"
assert quality["missing_order_date"] == 0, "Gold dashboard has missing order_date"

print("[OK] Gold dashboard table is non-empty, line-grain unique and date-complete")
print(dict(quality.asDict()))

### Beyond ad-hoc asserts: data quality frameworks

The checks above are deliberately simple `assert` statements, good for a live
demo but not how production Gold pipelines stay reliable over time. As the
checklist grows, teams typically move to a framework instead of copy-pasting
SQL asserts into every notebook:

| Approach | What it adds | When to reach for it |
|---|---|---|
| Ad-hoc SQL asserts (this notebook) | fast, readable, zero setup | demos, one-off validation, training |
| Delta `CONSTRAINT` (see Delta contract demo above) | enforced at write time, rejects bad rows | stable, table-level rules that should never be violated |
| **DQX** (Databricks Labs) | declarative, reusable expectations with pass/fail/quarantine outcomes, runs natively on Spark/UC | a Gold layer with many tables and a growing rule set |
| **Lakehouse Monitoring** | profiling, drift and anomaly detection over time, UC-native dashboards | tracking data quality trends across refreshes, not just one-time checks |

For this course, ad-hoc asserts are enough to prove the pattern. If Gold grows
beyond a handful of tables, point participants to DQX or Lakehouse Monitoring
instead of a hand-rolled assert library.

## 16. Optional bonus: SCD in analyst context

Slowly changing dimensions matter when a customer or product attribute changes
and historical reporting must preserve the old value. This is not a data
engineering detail only; it changes what Power BI users see in historical
trends and slicers.

Example: a customer moved from `SMB` to `Enterprise` in May. Should January
revenue move to Enterprise too, or should January remain SMB? That is a model
decision, not a visual formatting decision.

| Question | Type 1 tendency | Type 2 tendency |
|---|---|---|
| Does history matter? | no | yes |
| Example | correct a misspelled segment | customer moved between segments |
| BI impact | old reports change | old reports remain historically true |
| Complexity | lower | higher |

Keep this as a discussion or 15-minute extension. It should not distract from
the main goal: a stable Gold layer for BI.

## Summary and artefacts

Created or refreshed:

- `gold.dim_date`
- `gold.dim_product`
- `gold.dim_customer`
- `gold.fact_sales`
- `gold.fact_sales_dashboard`

Covered in this notebook:

- Lakehouse DW overview and Delta Lake foundations,
- Delta metadata inspection with `DESCRIBE DETAIL`, `DESCRIBE HISTORY`, time travel, constraints and schema evolution,
- Medallion compared with Kimball, Inmon, Data Vault and Data Mesh,
- Gold BI contract discussion,
- Kimball-style Gold model build,
- reconciliation, quality gates and data-quality risks (including a
  framework note: ad-hoc asserts vs Delta `CONSTRAINT` vs DQX vs Lakehouse
  Monitoring),
- SCD discussion (bonus).

Prepared for the next notebook:

- a trusted Gold detail contract (`gold.fact_sales_dashboard`) ready to shape
  into a Power BI semantic layer in `day1_03_gold_modeling_for_powerbi.ipynb`.